# 키워드 기반 배치 채점 노트북

`tests/` 폴더에서 `EXERCISE_KEYWORD`가 파일명에 포함된 영상을 전부 찾아 한 번에
채점합니다. `Test.ipynb`의 키워드 기반 배치 추출과 같은 방식입니다 — 영상 하나하나
`INPUT_PATH`를 바꿔가며 `score_new_video.ipynb`를 반복 실행할 필요 없이, 여러 영상의
결과를 표/그래프로 한 번에 비교할 수 있습니다.

**사전 준비물**: `models/{EXERCISE_KEYWORD}_reference.npz`, `data/processed/templates.npz`

**대상 파일**: `tests/` 폴더 안에서 파일명에 `EXERCISE_KEYWORD`가 포함된 `.mp4`와 `.npy`를
전부 찾습니다. `.mp4`인데 대응하는 `_keypoints.npy`가 아직 없으면 좌표 추출부터
자동으로 진행합니다 (이미 있으면 재추출하지 않고 재사용).

## 0. 환경 설정

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

project_root = Path(".").resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.pose_extraction.blazepose_extractor import BlazePoseExtractor, ExtractionConfig, PoseNotDetectedError
from src.scoring_model.score_reps import score_video, load_template

plt.rcParams["figure.figsize"] = (10, 4)
np.set_printoptions(precision=3, suppress=True)

# 한글 폰트가 없으면 그래프의 한글 라벨이 네모(□)로 깨질 수 있어, 있는 폰트 중 하나로 자동 설정
_korean_fonts = [f.name for f in fm.fontManager.ttflist if any(
    kw in f.name for kw in ["Malgun", "NanumGothic", "AppleGothic", "Noto Sans CJK", "Noto Sans KR"]
)]
if _korean_fonts:
    plt.rcParams["font.family"] = _korean_fonts[0]
plt.rcParams["axes.unicode_minus"] = False

## 1. 키워드 및 경로 설정

`EXERCISE_KEYWORD`를 바꾸면 대상 운동이 바뀝니다 (예: "pushup" -> "squat").

In [ ]:
EXERCISE_KEYWORD = "pushup"
TESTS_DIR = project_root / "tests"
MODEL_DIR = project_root / "models"
PROCESSED_DIR = project_root / "data" / "processed"
POSE_MODEL_PATH = "models/pose_landmarker_full.task"  # 좌표 추출이 필요한 .mp4가 있을 때만 사용

MIN_DISTANCE = 30
MIN_REP_FRAMES = 30

reference_path = MODEL_DIR / f"{EXERCISE_KEYWORD}_reference.npz"
if not reference_path.exists():
    raise FileNotFoundError(
        f"{reference_path} 가 없습니다. 먼저 '{EXERCISE_KEYWORD}' 기준 모델을 학습해두세요."
    )

template = load_template(PROCESSED_DIR, EXERCISE_KEYWORD)
print(f"'{EXERCISE_KEYWORD}' 템플릿 로드 완료 (길이 {len(template)})")

## 2. `tests/` 폴더에서 키워드 포함 영상 전부 탐색

`.mp4`와 `.npy`를 모두 찾되, 같은 영상의 `.mp4`+`.npy`가 둘 다 있으면 `.npy`(이미
추출된 것)를 우선 사용해서 중복 추출을 피합니다.

In [ ]:
mp4_files = sorted(
    p for p in TESTS_DIR.glob("*.mp4")
    if EXERCISE_KEYWORD.lower() in p.stem.lower()
)
npy_files = sorted(
    p for p in TESTS_DIR.glob("*_keypoints.npy")
    if EXERCISE_KEYWORD.lower() in p.stem.lower()
)
npy_stems = {p.stem for p in npy_files}

# mp4 중 이미 대응하는 keypoints.npy가 있는 것은 재추출 대상에서 제외
def _expected_npy_name(mp4_path: Path) -> str:
    return f"{EXERCISE_KEYWORD}_{mp4_path.stem}_keypoints"

mp4_needing_extraction = [p for p in mp4_files if _expected_npy_name(p) not in npy_stems]
mp4_already_extracted = [p for p in mp4_files if _expected_npy_name(p) in npy_stems]

print(f"🔍 '{EXERCISE_KEYWORD}' 키워드 포함 파일 발견:")
print(f"  - 이미 추출된 keypoints.npy: {len(npy_files)}개")
print(f"  - mp4 (이미 추출됨, npy 재사용): {len(mp4_already_extracted)}개")
print(f"  - mp4 (좌표 추출 필요): {len(mp4_needing_extraction)}개")

## 3. 필요한 것만 좌표 추출

In [ ]:
if mp4_needing_extraction:
    config = ExtractionConfig(model_path=POSE_MODEL_PATH, include_z=False, target_fps=None)
    for mp4_path in mp4_needing_extraction:
        output_path = mp4_path.parent / f"{_expected_npy_name(mp4_path)}.npy"
        try:
            with BlazePoseExtractor(config) as extractor:
                print(f"🔄 좌표 추출 중: {mp4_path.name}")
                keypoints, meta = extractor.extract_from_video(mp4_path)
            np.save(output_path, keypoints)
            print(f"   ↳ 완료! 검출률: {meta['detected_ratio']:.1%}")
            npy_files.append(output_path)
        except PoseNotDetectedError as e:
            print(f"   ↳ ⚠️ 스킵 (포즈 미검출): {e}")
else:
    print("추가로 추출할 영상이 없습니다.")

# 최종 채점 대상 목록: 이미 있던 npy + 방금 추출한 npy
all_keypoint_files = sorted(set(npy_files))
print(f"\n채점 대상 총 {len(all_keypoint_files)}개")

## 4. 전부 채점

각 영상을 `score_video()`로 채점하고, 하나의 표로 합칩니다. 영상 하나가 실패해도
전체가 멈추지 않도록 개별적으로 예외 처리합니다.

In [ ]:
all_results = []
failed = []

for keypoints_path in all_keypoint_files:
    try:
        results = score_video(
            keypoints_path=keypoints_path,
            exercise=EXERCISE_KEYWORD,
            template=template,
            model_dir=MODEL_DIR,
            min_distance=MIN_DISTANCE,
            min_rep_frames=MIN_REP_FRAMES,
        )
        all_results.extend(results)
    except Exception as e:
        print(f"❌ {keypoints_path.name} 채점 실패: {e}")
        failed.append((keypoints_path.name, str(e)))

if not all_results:
    raise RuntimeError("채점된 rep이 하나도 없습니다 — 위 실패 로그를 확인하세요.")

results_df = pd.DataFrame(all_results)
print(f"✅ 영상 {len(all_keypoint_files) - len(failed)}개, rep {len(results_df)}개 채점 완료"
      + (f" (실패 {len(failed)}개)" if failed else ""))
results_df[["video_id", "rep_idx", "score", "score_source", "distance"]]

## 5. 영상별 요약

In [ ]:
video_summary = results_df.groupby("video_id").agg(
    rep_수=("rep_idx", "count"),
    평균점수=("score", "mean"),
    최저점수=("score", "min"),
    규칙적용_비율=("score_source", lambda s: (s == "rule").mean()),
).round(1).sort_values("평균점수")

video_summary

## 6. 영상별 점수 분포 시각화

In [ ]:
video_ids = results_df["video_id"].unique()
fig, ax = plt.subplots(figsize=(max(10, len(video_ids) * 1.2), 5))

positions = range(len(video_ids))
for i, vid in enumerate(video_ids):
    sub = results_df[results_df["video_id"] == vid]
    colors = ["tab:orange" if s == "rule" else "tab:blue" for s in sub["score_source"]]
    jitter = np.linspace(-0.15, 0.15, len(sub)) if len(sub) > 1 else [0]
    ax.scatter([i + j for j in jitter], sub["score"], c=colors, s=60, zorder=3, edgecolors="white")

ax.set_xticks(list(positions))
ax.set_xticklabels(video_ids, rotation=30, ha="right")
ax.set_ylabel("score"); ax.set_ylim(0, 100)
ax.axhline(50, color="gray", linestyle="--", linewidth=1, zorder=1)
ax.set_title(f"'{EXERCISE_KEYWORD}' 영상별 rep 점수 (파랑=통계 기준, 주황=규칙 기반)")
plt.tight_layout()
plt.show()

## 7. 가장 낮은 점수 rep들 자연어 피드백

전체 영상 통틀어 점수가 가장 낮은 rep들만 모아서 무엇이 문제인지 보여줍니다.

In [ ]:
low_score_reps = results_df.sort_values("score").head(10)
print("=== 전체 영상 통틀어 점수가 낮은 rep TOP 10 ===\n")
for _, row in low_score_reps.iterrows():
    print(f"[{row['video_id']} / rep {row['rep_idx']}] score={row['score']:.1f} (source={row['score_source']})")
    if row["outlier_messages"]:
        for msg in row["outlier_messages"][:3]:
            print(f"   - {msg}")
    else:
        print("   - (이상치로 뚜렷하게 잡힌 특징 없음 — 규칙 기반 감점일 수 있음)")
    print()